In [ ]:
import pprint
import numpy as np
from pathlib import Path
import torch

import datasets
import evaluate

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


DATASET_NAME = "alexneo68/for_text_classification_food_not_food_v2"
MODEL_NAME = "DeepPavlov/rubert-base-cased-sentence"
MODEL_SAVE_DIR_NAME = "models/food_not_food_text_classifier-ru_deep_pavlov_rubert-base-cased"

model_save_dir = Path(MODEL_SAVE_DIR_NAME)
model_save_dir.mkdir(parents=True, exist_ok=True)

dataset = datasets.load_dataset(path=DATASET_NAME)

id2label = {0: "not_food", 1: "food"}
label2id = {"not_food": 0, "food": 1}

def map_labels_to_number(example):
    example['label'] = label2id[example["label"]]
    return example

dataset = dataset['train'].map(map_labels_to_number)


dataset = dataset.cast_column("label", datasets.Value("int64"))
assert dataset.features["label"] == datasets.Value("int64"), dataset.features["label"]


dataset = dataset.train_test_split(test_size=.2, seed=42)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME, use_fast=True)

def tokenize_caption(examples):
    return tokenizer(examples['caption'], padding=True, truncation=True)

tokenized_dataset = dataset.map(function=tokenize_caption, batched=True, batch_size=1000)

accuracy_metric = evaluate.load('accuracy')

def compute_accuracy(predictions_and_labels):

    predictions, labels = predictions_and_labels

    if len(predictions.shape) >= 2:
        predictions =  np.argmax(predictions, axis=1)


    return accuracy_metric.compute(predictions=predictions, references=labels)

print(f"Load model: {MODEL_NAME}")

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id    
)

print(f"Loading complete!")

trainings_arguments = TrainingArguments(
    output_dir=MODEL_SAVE_DIR_NAME,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none",
    push_to_hub=False,
    hub_private_repo=False
)


trainer = Trainer(
    model=model,
    args=trainings_arguments,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    processing_class=tokenizer,
    compute_metrics=compute_accuracy
)

print(f"Model is start training...")

results = trainer.train()

print(f"Model trained complete, saving to : {model_save_dir}")

trainer.save_model(output_dir=model_save_dir)










Load model: DeepPavlov/rubert-base-cased-sentence


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased-sentence
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading complete!
Model is start training...


/Users/makewww68/Documents/learning_projects/ml-text-classifications/.venv-hf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Model trained complete, saving to : models/food_not_food_text_classifier-ru_deep_pavlov_rubert-base-cased


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

uploading model to hf


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: Internal error: invalid shard failed to verify cas info for xorb a889a6949dcc084a49bd91cdc460ba6821f54c01fe59aa24124834e3344f0efc; xorb not found

In [11]:

print(f"uploading model to hf")

model_upload_url = trainer.push_to_hub(
    commit_message="Загружена модель обученная на русскоязычных данных"
)

print(f"Модель загружена по следующему пути: {model_upload_url}")

print(f"Проверяем модель на тестовом наборе")

predictions_all = trainer.predict(tokenized_dataset['test'])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f"Метрики предсказаний на тестовом наборе:")
pprint.pprint(prediction_metrics)

uploading model to hf


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель загружена по следующему пути: https://huggingface.co/alexneo68/food_not_food_text_classifier-ru_deep_pavlov_rubert-base-cased/commit/496176f767b4097a6bb38ea92d3ba568054a3fce
Проверяем модель на тестовом наборе


/Users/makewww68/Documents/learning_projects/ml-text-classifications/.venv-hf/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Метрики предсказаний на тестовом наборе:
{'test_accuracy': 0.9945553539019963,
 'test_loss': 0.03489519655704498,
 'test_runtime': 3.2025,
 'test_samples_per_second': 516.154,
 'test_steps_per_second': 16.237}


In [29]:
from typing import Dict

def classifier_food_not_food(sentence: str) -> Dict[str, float]:

    food_not_food_classifier = pipeline(
        task="text-classification",
        model=model_save_dir,
        device=torch.device("mps") if torch.backends.mps.is_available() else "cpu",
        batch_size=32, 
        top_k=None
    )

    outputs = food_not_food_classifier(sentence)[0]

    output_dict = {}

    for o in outputs:
        output_dict[o['label']] = o['score']

    return output_dict

In [30]:
classifier_food_not_food('жареная картошка')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'food': 0.9998989105224609, 'not_food': 0.00010105455294251442}

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=classifier_food_not_food,
    inputs="text",
    outputs=gr.Label(num_top_classes=2),
    title="Классификатор - еда/не еда",
    description="Это текстовый классификатор, который соотносит введенное предложение к классу еды или не еды",
    examples=[
        ["Это арбуз довольно зрелый"],
        ["На тарелке лежали два апельсина"],
    ]
)

In [32]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Created dataset file at: .gradio/flagged/dataset1.csv


In [33]:
from pathlib import Path

demos_dir = Path('./demos')
demos_dir.mkdir(exist_ok=True)

f_n_f_text_classifier_demo_dir = Path(demos_dir, "food_not_food_text_classifier")
f_n_f_text_classifier_demo_dir.mkdir(exist_ok=True)



In [ ]:
%%writefile ./demos/food_not_food_text_classifier/app.py

import torch
import gradio as gr

from typing import Dict
from transformers import pipeline

def classifier_food_not_food(sentence: str) -> Dict[str, float]:

    food_not_food_classifier = pipeline(
        task="text-classification",
        model=model_save_dir,
        device=torch.device("mps") if torch.backends.mps.is_available() else "cpu",
        batch_size=32, 
        top_k=None
    )

    outputs = food_not_food_classifier(sentence)[0]

    output_dict = {}

    for o in outputs:
        output_dict[o['label']] = o['score']

    return output_dict


description = """
Это текстовый классификатор позволяющий относить введенное предложение к одному из двух типов, идет ли в нем речь о еде или не о еде
Используется зафайнтьюненая модель - https://huggingface.co/alexneo68/food_not_food_text_classifier-ru_deep_pavlov_rubert-base-cased
Исходный код - https://github.com/AlexNeo68/ml-practice/blob/main/text_classification_ru.ipynb
"""

